# IoT: cảm biến chất lượng không khí UCI (Ý, 03/2004 - 04/2005)

Một trạm đặt giữa đường phố một thành phố Ý ghi **mỗi giờ** hai loại số đo:

- `PT08.S*`: 5 cảm biến oxit kim loại **giá rẻ** (đơn vị là tín hiệu thô của cảm biến);
- `*(GT)`: nồng độ "ground truth" từ máy phân tích chuẩn đặt cạnh (CO mg/m³, NOx ppb, NO2 µg/m³, C6H6 µg/m³);
- `T`, `RH`, `AH`: nhiệt độ, độ ẩm tương đối, độ ẩm tuyệt đối.

Đây là dữ liệu IoT thật với mọi tật xấu: dấu `;`, **dấu phẩy thập phân**, ngày `10/03/2004` và giờ
`18.00.00` tách hai cột, hai cột rỗng thừa ở cuối, hàng trăm dòng rỗng cuối tệp, và giá trị thiếu
mã hoá là **`-200`**. Lấy trung bình thẳng trên `-200` sẽ ra nồng độ âm.

Việc của bạn: `load_raw` → `clean` → `outages`, `monthly_profile`, `insights` → biểu đồ →
`write_clean`. Chấm:
`uv run pytest tracks/data-engineer/modules/00-introduction/exercises/basic/03-iot-air-quality -q`

In [ ]:
# explore: tìm gốc repo (thư mục có pyproject.toml), rồi tải bộ dữ liệu nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

RAW = ensure("air-quality-uci")
CLEAN = ROOT / "data" / "clean" / "air-quality-uci"
print(RAW, sorted(p.name for p in RAW.iterdir()))

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

# Tên cột gốc -> tên gọn, snake_case. clean() trả các cột theo đúng thứ tự này, sau cột ts.
COLUMNS = {
    "CO(GT)": "co_gt", "PT08.S1(CO)": "s1_co", "NMHC(GT)": "nmhc_gt", "C6H6(GT)": "c6h6_gt",
    "PT08.S2(NMHC)": "s2_nmhc", "NOx(GT)": "nox_gt", "PT08.S3(NOx)": "s3_nox", "NO2(GT)": "no2_gt",
    "PT08.S4(NO2)": "s4_no2", "PT08.S5(O3)": "s5_o3", "T": "temp_c", "RH": "rh_pct", "AH": "abs_humidity",
}
SENSORS = ["s1_co", "s2_nmhc", "s3_nox", "s4_no2", "s5_o3"]   # 5 cảm biến giá rẻ
SENTINEL = -200

## 1. Đọc tệp thô

`load_raw(zip_path)` đọc `AirQualityUCI.csv` **ngay trong tệp zip** (không giải nén ra đĩa), với
`sep=";"` và `decimal=","`. Trả DataFrame nguyên trạng: chưa đổi tên, chưa bỏ dòng nào.

In [ ]:
def load_raw(zip_path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path) as zf, zf.open("AirQualityUCI.csv") as fh:
        return pd.read_csv(fh, sep=";", decimal=",")

In [ ]:
# explore: nhìn dữ liệu thô trước khi tin tài liệu
raw = load_raw(RAW / "air-quality.zip")
print(raw.shape)
print((raw == -200).sum())
raw.tail(3)

## 2. Làm sạch

`clean(raw)` trả DataFrame cột `ts` + các cột trong `COLUMNS` (đã đổi tên, đúng thứ tự):

- bỏ dòng rỗng hoàn toàn và cột rỗng thừa (`Unnamed: 15`, `Unnamed: 16`);
- `ts` = `Date` + `Time` (`"10/03/2004"` + `"18.00.00"`), kiểu datetime, giờ địa phương không kèm múi giờ (tệp không ghi múi giờ);
- **đúng** giá trị `-200` thành null. Giá trị âm khác (nhiệt độ `-1,5`) là thật, giữ nguyên;
- một giờ chỉ một dòng (giữ dòng đầu), sắp theo `ts`.

In [ ]:
def clean(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.dropna(how="all").dropna(axis=1, how="all")
    ts = pd.to_datetime(df["Date"] + " " + df["Time"], format="%d/%m/%Y %H.%M.%S")
    values = df[list(COLUMNS)].rename(columns=COLUMNS).astype("float64").mask(lambda x: x == SENTINEL)
    out = pd.concat([ts.rename("ts"), values], axis=1)
    return out.drop_duplicates("ts").sort_values("ts").reset_index(drop=True)

In [ ]:
# explore: tỉ lệ thiếu theo cột sau khi -200 thành null
df = clean(raw)
df.isna().mean().sort_values(ascending=False).round(3)

## 3. Mất kết nối và hồ sơ theo tháng

- `outages(df)`: các **đợt** liên tiếp mà cả 5 cảm biến `SENSORS` cùng null (thiết bị mất điện/mất
  kết nối, khác với một máy phân tích hỏng). Trả DataFrame cột `start` (giờ đầu đợt), `hours` (số giờ), sắp theo `start`.
- `monthly_profile(df)`: index là tháng (`Period[M]`, từ `ts.dt.to_period("M")`), các cột `days`
  (số ngày khác nhau có dòng), trung bình `co_gt`, `nox_gt`, `no2_gt`, `c6h6_gt`, `temp_c` (bỏ qua null),
  và `co_missing_share` (tỉ lệ giờ thiếu `co_gt`).

In [ ]:
def outages(df: pd.DataFrame) -> pd.DataFrame:
    down = df[SENSORS].isna().all(axis=1)
    run_id = (down != down.shift()).cumsum()            # đánh số mỗi đoạn liên tiếp cùng trạng thái
    runs = df.loc[down].groupby(run_id[down])["ts"].agg(start="min", hours="size")
    return runs.sort_values("start").reset_index(drop=True)


def monthly_profile(df: pd.DataFrame) -> pd.DataFrame:
    month = df["ts"].dt.to_period("M")
    out = df.groupby(month)[["co_gt", "nox_gt", "no2_gt", "c6h6_gt", "temp_c"]].mean()
    out.insert(0, "days", df.groupby(month)["ts"].apply(lambda s: s.dt.date.nunique()))
    out["co_missing_share"] = df["co_gt"].isna().groupby(month).mean()
    out.index.name = "month"
    return out

## 4. Insight

`insights(df) -> dict` với đúng các khoá:

| Khoá | Nghĩa |
| --- | --- |
| `rows`, `start`, `end` | số dòng, giờ đầu và cuối (`"YYYY-MM-DD HH:MM"`) |
| `missing_share` | dict tên cột → tỉ lệ null (mọi cột trừ `ts`) |
| `nmhc_last_valid` | ngày (`"YYYY-MM-DD"`) cuối cùng `nmhc_gt` còn có giá trị |
| `outage_hours`, `longest_outage_hours`, `longest_outage_start` | tổng giờ mất kết nối (theo `outages`), đợt dài nhất và giờ bắt đầu |
| `co_peak_hour`, `co_morning_peak_hour`, `co_trough_hour` | giờ (0–23) có CO trung bình cao nhất cả ngày / cao nhất trong 0–11 / thấp nhất |
| `co_weekday_mean`, `co_weekend_mean`, `weekend_drop_pct` | CO trung bình thứ 2–6, thứ 7–CN, và cuối tuần thấp hơn bao nhiêu % |
| `lowest_co_month` | tháng (`"YYYY-MM"`) CO trung bình thấp nhất, **chỉ xét tháng có ít nhất 28 ngày dữ liệu**; không có thì `None` |

In [ ]:
def insights(df: pd.DataFrame) -> dict:
    fmt = "%Y-%m-%d %H:%M"
    down = outages(df)
    longest = down.loc[down["hours"].idxmax()] if len(down) else None
    by_hour = df.groupby(df["ts"].dt.hour)["co_gt"].mean()
    weekend = df["ts"].dt.day_of_week >= 5
    weekday_mean = df.loc[~weekend, "co_gt"].mean()
    weekend_mean = df.loc[weekend, "co_gt"].mean()
    months = monthly_profile(df)
    full = months[months["days"] >= 28]
    return {
        "rows": len(df),
        "start": df["ts"].min().strftime(fmt),
        "end": df["ts"].max().strftime(fmt),
        "missing_share": df.drop(columns="ts").isna().mean().to_dict(),
        "nmhc_last_valid": df.loc[df["nmhc_gt"].notna(), "ts"].max().strftime("%Y-%m-%d"),
        "outage_hours": int(down["hours"].sum()),
        "longest_outage_hours": int(longest["hours"]) if longest is not None else 0,
        "longest_outage_start": longest["start"].strftime(fmt) if longest is not None else None,
        "co_peak_hour": int(by_hour.idxmax()),
        "co_morning_peak_hour": int(by_hour.loc[0:11].idxmax()),
        "co_trough_hour": int(by_hour.idxmin()),
        "co_weekday_mean": float(weekday_mean),
        "co_weekend_mean": float(weekend_mean),
        "weekend_drop_pct": float((1 - weekend_mean / weekday_mean) * 100),
        "lowest_co_month": str(full["co_gt"].idxmin()) if len(full) else None,
    }

In [ ]:
# explore: các con số để trả lời trắc nghiệm
INSIGHTS = insights(df)
print(outages(df).sort_values("hours", ascending=False).head())
INSIGHTS

## 5. Biểu đồ

Vẽ ít nhất hai biểu đồ (mỗi biểu đồ một cell `explore`):

1. CO trung bình theo giờ trong ngày, ngày thường vs cuối tuần: hai đỉnh nói gì về nguồn ô nhiễm?
2. Bản đồ nhiệt **tỉ lệ thiếu** theo tháng × cột: thiếu ngẫu nhiên hay có cấu trúc?
3. (tuỳ chọn) NOx và CO theo tháng: mùa nào ô nhiễm hơn, tháng 8 có gì lạ?

In [ ]:
# explore: CO theo giờ, ngày thường vs cuối tuần
import matplotlib.pyplot as plt

weekend = df["ts"].dt.day_of_week >= 5
co = df.groupby([weekend, df["ts"].dt.hour])["co_gt"].mean().unstack(0)
ax = co.rename(columns={False: "thứ 2-6", True: "thứ 7-CN"}).plot(figsize=(9, 3.5), marker="o")
ax.set_title("CO(GT) trung bình theo giờ: hai đỉnh giờ đi làm và tan tầm")
ax.set_xlabel("giờ")
ax.set_ylabel("CO mg/m³")
plt.show()

In [ ]:
# explore: tỉ lệ thiếu theo tháng x cột
miss = df.drop(columns="ts").isna().groupby(df["ts"].dt.to_period("M")).mean()
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(miss.T.values, aspect="auto", cmap="Reds", vmin=0, vmax=1)
ax.set_yticks(range(len(miss.columns)), miss.columns)
ax.set_xticks(range(len(miss.index)), miss.index.astype(str), rotation=45)
fig.colorbar(im, label="tỉ lệ thiếu")
ax.set_title("Thiếu có cấu trúc: NMHC chết sau 04/2004, máy phân tích GT hỏng theo đợt")
plt.show()

In [ ]:
# explore: NOx và CO theo tháng
months = monthly_profile(df)
fig, ax1 = plt.subplots(figsize=(10, 3.5))
ax1.bar(months.index.astype(str), months["nox_gt"], color="tab:gray", label="NOx ppb")
ax2 = ax1.twinx()
ax2.plot(months.index.astype(str), months["co_gt"], color="tab:red", marker="o", label="CO mg/m³")
ax1.set_ylabel("NOx ppb")
ax2.set_ylabel("CO mg/m³")
ax1.tick_params(axis="x", rotation=45)
ax1.set_title("NOx nhảy bậc từ 09/2004 trong khi CO chỉ nhích; tháng 8 (Ferragosto) thấp nhất")
plt.show()

## 6. Ghi dữ liệu sạch

`write_clean(df, out_dir)` ghi đè `out_dir/hourly.parquet` (qua tệp tạm) và trả đường dẫn.

In [ ]:
def write_clean(df: pd.DataFrame, out_dir: Path) -> Path:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / "hourly.parquet"
    tmp = out_dir / ".hourly.parquet.tmp"
    df.to_parquet(tmp, index=False)
    tmp.replace(path)
    return path

In [ ]:
# explore: ghi data/clean/air-quality-uci/hourly.parquet
path = write_clean(df, CLEAN)
print(path, len(pd.read_parquet(path)), "dòng")

## 7. Báo cáo bài học

Làm phần trắc nghiệm trong app sau khi chạy notebook.